In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"../data/superstore_raw.csv")
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,8/11/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,8/11/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,12/6/2016,16/6/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,11/10/2015,18/10/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,11/10/2015,18/10/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [20]:
df = df.rename(columns={
    "Postal Code": "postal_code",
    "Country": "country",
    "Region": "region",
    "State": "state",
    "City": "city",
    "Order ID": "order_id",
    "Order Date": "order_date",
    "Ship Date": "ship_date",
    "Ship Mode": "ship_mode",
    "Customer ID": "customer_id",
    "Product ID": "product_id",
    "Sales": "sales",
    "Quantity": "quantity",
    "Discount": "discount",
    "Profit": "profit"
})

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   order_id       9994 non-null   object 
 2   order_date     9994 non-null   object 
 3   ship_date      9994 non-null   object 
 4   ship_mode      9994 non-null   object 
 5   customer_id    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   country        9994 non-null   object 
 9   city           9994 non-null   object 
 10  state          9994 non-null   object 
 11  postal_code    9994 non-null   int64  
 12  region         9994 non-null   object 
 13  product_id     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  sales          9994 non-null   float64
 18  quantity

In [4]:
dim_customer = df[["Customer ID", "Customer Name", "Segment"]]

dim_customer = dim_customer.rename(columns={
    "Customer ID": "customer_id",
    "Customer Name": "customer_name",
    "Segment": "segment"
})

dim_customer = dim_customer.drop_duplicates()

In [5]:
dim_customer["customer_id"].duplicated().sum()

np.int64(0)

In [6]:
dim_customer.shape

(793, 3)

In [7]:
dim_product = df[["Product ID", "Product Name", "Category", "Sub-Category"]]

dim_product = dim_product.rename(columns={
    "Product ID": "product_id",
    "Product Name": "product_name",
    "Category": "category",
    "Sub-Category": "sub_category"
})

dim_product = dim_product.drop_duplicates()

In [8]:
dim_product.shape

(1894, 4)

In [9]:
dim_product["product_id"].duplicated().sum()

np.int64(32)

In [10]:
dim_product[dim_product["product_id"].duplicated(keep=False)].sort_values("product_id")

,product_id,product_name,category,sub_category
2471,FUR-BO-10002213,"Sauder Forest Hills Library, Woodland Oak Finish",Furniture,Bookcases
2115,FUR-BO-10002213,DMI Eclipse Executive Suite Bookcases,Furniture,Bookcases
66,FUR-CH-10001146,"Global Value Mid-Back Manager's Chair, Gray",Furniture,Chairs
128,FUR-CH-10001146,"Global Task Chair, Black",Furniture,Chairs
1459,FUR-FU-10001473,DAX Wood Document Frame,Furniture,Furnishings
...,...,...,...,...
1219,TEC-PH-10002200,Samsung Galaxy Note 2,Technology,Phones
2596,TEC-PH-10002310,Plantronics Calisto P620-M USB Wireless Speake...,Technology,Phones
1378,TEC-PH-10002310,Panasonic KX T7731-B Digital phone,Technology,Phones
922,TEC-PH-10004531,OtterBox Commuter Series Case - iPhone 5 & 5s,Technology,Phones


In [11]:
# Resolved inconsistent product records by deduplicating on product_id 
# and keeping the first observed product attributes.
dim_product = dim_product.drop_duplicates(subset=["product_id"])

In [12]:
dim_product["product_id"].duplicated().sum()

np.int64(0)

In [13]:
dim_product.shape

(1862, 4)

In [14]:
dim_location = df[["Postal Code", "Country", "Region", "State", "City"]]

dim_location = dim_location.rename(columns={
    "Postal Code": "postal_code",
    "Country": "country",
    "Region": "region",
    "State": "state",
    "City": "city"
})

dim_location["postal_code"] = dim_location["postal_code"].astype(str)

dim_location = dim_location.drop_duplicates()

In [15]:
dim_location.shape

(632, 5)

In [16]:
dim_location = dim_location.reset_index(drop=True)
dim_location

,postal_code,country,region,state,city
0,42420,United States,South,Kentucky,Henderson
1,90036,United States,West,California,Los Angeles
2,33311,United States,South,Florida,Fort Lauderdale
3,90032,United States,West,California,Los Angeles
4,28027,United States,South,North Carolina,Concord
...,...,...,...,...,...
627,72762,United States,South,Arkansas,Springdale
628,95240,United States,West,California,Lodi
629,77571,United States,Central,Texas,La Porte
630,45040,United States,East,Ohio,Mason


In [17]:
dim_location["location_id"] = dim_location.index + 1

In [23]:
df["postal_code"] = df["postal_code"].astype(str)

In [24]:
df = df.merge(
    dim_location,
    on=["postal_code", "country", "region", "state", "city"],
    how="left"
)

In [26]:
df["location_id"].isnull().sum()

np.int64(0)

In [27]:
fact_sales = df[[
    "order_id",
    "order_date",
    "ship_date",
    "ship_mode",
    "customer_id",
    "product_id",
    "location_id",
    "sales",
    "quantity",
    "discount",
    "profit"
]]

In [28]:
fact_sales[["order_id", "product_id"]].duplicated().sum()

np.int64(8)

In [29]:
fact_sales[
    fact_sales[["order_id","product_id"]].duplicated(keep=False)
].sort_values(["order_id","product_id"])

,order_id,order_date,ship_date,ship_mode,customer_id,product_id,location_id,sales,quantity,discount,profit
6498,CA-2015-103135,24/7/2015,28/7/2015,Standard Class,SS-20515,OFF-BI-10000069,176,135.090,9,0.0,62.1414
6500,CA-2015-103135,24/7/2015,28/7/2015,Standard Class,SS-20515,OFF-BI-10000069,176,90.060,6,0.0,41.4276
350,CA-2016-129714,1/9/2016,3/9/2016,First Class,AB-10060,OFF-PA-10001970,30,24.560,2,0.0,11.5432
352,CA-2016-129714,1/9/2016,3/9/2016,First Class,AB-10060,OFF-PA-10001970,30,49.120,4,0.0,23.0864
1300,CA-2016-137043,23/12/2016,25/12/2016,Second Class,LC-17140,FUR-FU-10003664,29,572.760,6,0.0,166.1004
1301,CA-2016-137043,23/12/2016,25/12/2016,Second Class,LC-17140,FUR-FU-10003664,29,286.380,3,0.0,83.0502
9168,CA-2016-140571,15/3/2016,19/3/2016,Standard Class,SJ-20125,OFF-PA-10001954,130,319.760,14,0.0,147.0896
9169,CA-2016-140571,15/3/2016,19/3/2016,Standard Class,SJ-20125,OFF-PA-10001954,130,45.680,2,0.0,21.0128
7881,CA-2017-118017,3/12/2017,6/12/2017,Second Class,LC-16870,TEC-AC-10002006,462,76.752,6,0.2,10.5534
7882,CA-2017-118017,3/12/2017,6/12/2017,Second Class,LC-16870,TEC-AC-10002006,462,102.336,8,0.2,14.0712


In [30]:
fact_sales = fact_sales.groupby(
    ["order_id", "product_id"],
    as_index=False
).agg({
    "order_date": "first",
    "ship_date": "first",
    "ship_mode": "first",
    "customer_id": "first",
    "location_id": "first",
    "discount": "first",
    "quantity": "sum",
    "sales": "sum",
    "profit": "sum"
})

In [31]:
fact_sales[["order_id","product_id"]].duplicated().sum()

np.int64(0)

In [32]:
fact_sales.shape

(9986, 11)

In [34]:
fact_sales.head()

,order_id,product_id,order_date,ship_date,ship_mode,customer_id,location_id,discount,quantity,sales,profit
0,CA-2014-100006,TEC-PH-10002075,7/9/2014,13/9/2014,Standard Class,DK-13375,24,0.0,3,377.970,109.6113
1,CA-2014-100090,FUR-TA-10003715,8/7/2014,12/7/2014,Standard Class,EB-13705,35,0.2,3,502.488,-87.9354
2,CA-2014-100090,OFF-BI-10001597,8/7/2014,12/7/2014,Standard Class,EB-13705,35,0.2,6,196.704,68.8464
3,CA-2014-100293,OFF-PA-10000176,14/3/2014,18/3/2014,Standard Class,NF-18475,134,0.2,6,91.056,31.8696
4,CA-2014-100328,OFF-BI-10000343,28/1/2014,3/2/2014,Standard Class,JC-15340,24,0.2,1,3.928,1.3257


In [35]:
dim_customer.to_csv("../data/dim_customer.csv", index=False)
dim_product.to_csv("../data/dim_product.csv", index=False)
dim_location.to_csv("../data/dim_location.csv", index=False)
fact_sales.to_csv("../data/fact_sales.csv", index=False)